# 🔍 RAG Query Notebook
Quick iteration on LightRAG queries. All tuneable knobs are in **Cell 1**.

In [ ]:
# ── Imports & shared setup ────────────────────────────────────────────────────
import sys, os
NOTEBOOK_DIR = os.path.abspath(".")          # notebooks/
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from rag_config import build_rag, LLM_MODEL, EMBEDDING_MODEL, LLM_BASE_URL, EMBED_BASE_URL, WORKING_DIR, LLM_MAX_TOKENS, LLM_TEMPERATURE
from lightrag import QueryParam

# ══════════════════════════════════════════════════════
# KNOBS — change these without restarting the kernel
# ══════════════════════════════════════════════════════
QUERY = "What is the function of the aorta?"
MODE  = "hybrid"    # naive | local | global | hybrid
# ══════════════════════════════════════════════════════

print(f"LLM          : {LLM_MODEL}")
print(f"Embed        : {EMBEDDING_MODEL}")
print(f"LLM URL      : {LLM_BASE_URL}")
print(f"Embed URL    : {EMBED_BASE_URL}")
print(f"Working dir  : {WORKING_DIR}")
print(f"max_tokens   : {LLM_MAX_TOKENS}  temperature: {LLM_TEMPERATURE}")

In [ ]:
# ── Build & initialise RAG (run once per kernel session) ──────────────────────
rag = build_rag()
await rag.initialize_storages()
print("RAG ready ✓")

In [ ]:
# ── Run query ─────────────────────────────────────────────────────────────────
# Edit QUERY / MODE in Cell 1, then re-run just this cell.
result = await rag.aquery(QUERY, param=QueryParam(mode=MODE))

print(f"Q [{MODE}]: {QUERY}")
print("-" * 60)
print(result)

In [ ]:
# ── Compare modes side-by-side ────────────────────────────────────────────────
MULTI_QUERY = QUERY   # override here if you want a different question
MODES_TO_COMPARE = ["naive", "local", "global", "hybrid"]

for m in MODES_TO_COMPARE:
    print(f"\n{'='*60}")
    print(f"[{m.upper()}] {MULTI_QUERY}")
    print('='*60)
    r = await rag.aquery(MULTI_QUERY, param=QueryParam(mode=m))
    print(r)

In [ ]:
# ── Sanity check: direct LLM call (no RAG) ───────────────────────────────────
from openai import AsyncOpenAI

client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key="none")

DIRECT_PROMPT  = QUERY
DIRECT_SYSTEM  = "You are a helpful medical assistant."

resp = await client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": DIRECT_SYSTEM},
        {"role": "user",   "content": DIRECT_PROMPT},
    ],
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
)

print(f"[Direct LLM — no RAG] {DIRECT_PROMPT}")
print("-" * 60)
print(resp.choices[0].message.content)
print(f"\nTokens: prompt={resp.usage.prompt_tokens}  completion={resp.usage.completion_tokens}")